# Session 5: 実機で PID を感じる — 角速度制御
# Session 5: Feel PID on Real Hardware — Angular Rate Control

## 目標 / Objective

レート PID ゲインを変更し、ステップ応答を比較して PID パラメータの効果を体感する。

Change rate PID gains, compare step responses, and feel the effect of PID parameters.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 角速度制御 | ドローンの最内ループ |
| ステップ応答テスト | `sf step` コマンド |
| ゲイン調整の効果 | Kp, Ti, Td の実機効果 |
| 性能指標の比較 | 立ち上がり時間、オーバーシュート |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stampfly_edu import load_sample_data, plot_step_response, compare_logs
from stampfly_edu.sim import DronePlant, simulate_pid, compare_controllers
from stampfly_edu.sim.simulate import compute_metrics

print("Ready! / 準備完了！")

## 1. 角速度制御ループの構造 / Rate Control Loop Structure

### 理論 / Theory

ドローンの姿勢制御は **カスケード構造** になっています：

```
                    Inner Loop (Rate) — Session 5 のテーマ
                    ┌──────────────────────────────────┐
角度目標 → [角度PID] → 角速度目標 → [レートPID] → モーター → ドローン → 角速度
                                      ↑                              │
                                      └──────── ジャイロセンサ ←──────┘
```

レート PID はジャイロセンサ（400Hz）からの角速度フィードバックを使用：

$$\tau = K_p \left( e_{\omega} + \frac{1}{T_i} \int e_{\omega} dt + \frac{T_d s}{\eta T_d s + 1} e_{\omega} \right)$$

### StampFly の角速度ダイナミクス

ロール軸の簡略モデル:

$$I_{xx} \dot{\omega} = \tau_{motor} - D_{rot} \omega$$

| パラメータ | 値 | 単位 |
|-----------|-----|------|
| $I_{xx}$ (ロール慣性モーメント) | $9.16 \times 10^{-6}$ | kg$\cdot$m$^2$ |
| モータ時定数 $\tau_m$ | 0.02 | s |
| アーム長 $L$ | 0.023 | m |

## 2. シミュレーションで予習 / Simulation Preview

In [ ]:
# Simulate rate PID with different gains
# 異なるゲインでレートPIDをシミュレーション
plant = DronePlant(axis="roll")
print(f"Plant: {plant}")

configs = [
    {"Kp": 0.1, "Ti": 0.0, "Td": 0.0},   # Low Kp, P only
    {"Kp": 0.5, "Ti": 0.0, "Td": 0.0},   # Medium Kp
    {"Kp": 0.5, "Ti": 0.5, "Td": 0.0},   # PI
    {"Kp": 0.5, "Ti": 0.5, "Td": 0.01},  # PID
]
labels = ["P(Kp=0.1)", "P(Kp=0.5)", "PI(Ti=0.5)", "PID(Td=0.01)"]

fig = compare_controllers(
    plant, configs, labels=labels,
    title="Rate PID Simulation (Roll Axis) / レートPIDシミュレーション (ロール軸)",
    setpoint=1.0,  # 1 rad/s target
    duration=0.5,   # 500ms
    dt=0.0025,      # 400Hz
)
plt.show()

## 3. サンプルデータで分析 / Analysis with Sample Data

実機のステップ応答データを分析します。
実機データがない場合はサンプルデータを使用します。

In [ ]:
# Load sample rate step response
# サンプルレートステップ応答を読み込む
log = load_sample_data("rate_step")

print(f"Columns: {list(log.columns)}")
print(f"Duration: {log['time'].iloc[-1]:.2f} s")
print(f"Sample rate: {1.0 / (log['time'].iloc[1] - log['time'].iloc[0]):.0f} Hz")
log.head()

In [ ]:
# Plot step response
# ステップ応答をプロット
ax = plot_step_response(
    log,
    output_col="p",
    setpoint_col="setpoint",
    title="Rate Step Response (Roll) / レートステップ応答 (ロール)",
)
plt.show()

In [ ]:
# Plot motor duty cycles during step
# ステップ中のモーターデューティ比をプロット
motor_cols = [c for c in log.columns if c.startswith('m')]
if motor_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    for col in motor_cols:
        ax.plot(log['time'], log[col], label=col, linewidth=1.0)
    ax.set_xlabel('Time (s) / 時間')
    ax.set_ylabel('Motor duty / モーターデューティ')
    ax.set_title('Motor Response / モーター応答')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. 実機実験手順 / Real Hardware Experiment

### sf step コマンドの使い方

ターミナルで以下のコマンドを実行：

```bash
# Roll axis step response test (amplitude 0.5 rad/s, duration 2s)
# ロール軸ステップ応答テスト
sf step rate roll --amplitude 0.5 --duration 2.0

# Pitch axis
# ピッチ軸
sf step rate pitch --amplitude 0.5 --duration 2.0
```

### 実験計画

1. デフォルトゲインでステップ応答を記録
2. Kp を 50% 増やして再テスト
3. Kp を 50% 減らして再テスト
4. Ti を追加して再テスト

In [ ]:
# Load and compare real experiment data (replace paths with actual files)
# 実験データの読み込みと比較（パスを実際のファイルに置き換えてください）

# Example: compare two log files
# 例: 2つのログファイルを比較
#
# log_default = load_flight_log("step_response_default.csv")
# log_high_kp = load_flight_log("step_response_high_kp.csv")
# fig = compare_logs(log_default, log_high_kp,
#                    columns=["p"],
#                    labels=("Default Kp", "High Kp"))
# plt.show()

print("Uncomment the code above and replace with your actual log file paths")
print("上のコードのコメントを外して、実際のログファイルパスに置き換えてください")

## 5. 性能指標の比較表 / Performance Metrics Table

実験結果を以下の表にまとめましょう。

In [ ]:
# Performance metrics table template
# 性能指標表テンプレート

# Fill in with your experimental results
# 実験結果を記入してください
results_table = pd.DataFrame({
    "Configuration / 設定": ["Default", "Kp x 1.5", "Kp x 0.5", "PI"],
    "Kp": [0.65, 0.975, 0.325, 0.65],
    "Ti": [0.0, 0.0, 0.0, 0.5],
    "Td": [0.01, 0.01, 0.01, 0.01],
    "Rise Time (s) / 立ち上がり": ["___", "___", "___", "___"],
    "Overshoot (%) / OS": ["___", "___", "___", "___"],
    "Settling (s) / 整定": ["___", "___", "___", "___"],
})

results_table

## 6. 考察課題 / Discussion Questions

1. **シミュレーションと実機の違い**: 応答のどこが違いましたか？その原因は？
2. **軸による違い**: ロールとピッチで応答が異なる場合、その理由は？（ヒント: 慣性モーメント）
3. **制御帯域**: レート PID の帯域はどのくらいか？より速い応答のために何が制約になるか？

---

1. **Simulation vs Reality**: Where did the responses differ? What causes the difference?
2. **Axis Differences**: If roll and pitch responses differ, why? (Hint: moments of inertia)
3. **Control Bandwidth**: What is the rate PID bandwidth? What limits faster response?

In [ ]:
# Your analysis here / ここで分析してみてください
